In [1]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F



In [2]:
# Pseudo code
vocab_size_4 = 4
id2word = ['<s>', 'a', 'b', 'c']
word2id = {
    '<s>': 0,
    'a': 1,
    'b': 2,
    'c': 3
}



In [3]:
print(id2word[0])
print(word2id['<s>'])

<s>
0


In [3]:
embed_dim_64 = 64

In [4]:
data = "aabbccaabbcc"

# Tokenize data + prepand a <s> token
input = torch.tensor([0,1,1,2,2,3,3,1,1,2,2,3,3])
actual = torch.tensor([1,1,2,2,3,3,1,1,2,2,3,3,1])
# actual_probs = torch.tensor([
#    # s  a  b  c
#    #[1, 0, 0, 0 ] <- this is shifted out of view in prediction
#     [0, 1, 0, 0 ],
#     [0, 1, 0, 0 ],
#     [0, 0, 1, 0 ],
#     [0, 0, 1, 0 ],
#     [0, 0, 0, 1 ],
#     [0, 0, 0, 1 ], 
#     [0, 1, 0, 0 ],
#     [0, 1, 0, 0 ],
#     [0, 0, 1, 0 ],
#     [0, 0, 1, 0 ],
#     [0, 0, 0, 1 ],
#     [0, 0, 0, 1 ], 
#     [0, 1, 0, 0 ]  # add the final predicted token — back to 'a'
# ]).float() # l = 13


In [5]:

embeddings_matrix = nn.Embedding(vocab_size_4, embed_dim_64)

W_Q = nn.Linear(embed_dim_64, embed_dim_64) # (linear layers)
W_K = nn.Linear(embed_dim_64, embed_dim_64)
W_V = nn.Linear(embed_dim_64, embed_dim_64)


In [6]:

seq_vectors = embeddings_matrix(input) # (13, 64)
seq_vectors.shape

torch.Size([13, 64])

In [ ]:
q = W_Q(seq_vectors)
k = W_K(seq_vectors)
v = W_V(seq_vectors)
print(q.shape, k.shape, v.shape) # (13, 64) each

In [ ]:
atn = q @ k.T
atn = atn / math.sqrt(embed_dim_64) # Not in Bes's code
atn.shape # (13,13)
print(atn)

In [13]:
neginf = torch.full_like(atn, float('-inf'))
mask = torch.triu(neginf, diagonal=1)

In [ ]:
atn_masked = atn + mask
print(atn_masked)

In [ ]:
atn_probs = F.softmax(atn_masked, dim=-1) # softmax along rows
print(atn_probs)

In [ ]:
out = atn_probs @ v
out.shape

In [15]:
ff = nn.Sequential(
    nn.Linear(embed_dim_64, embed_dim_64 * 4),
    nn.ReLU(),
    nn.Linear(embed_dim_64 * 4, embed_dim_64)
)

In [ ]:
hidden = ff(out)
hidden.shape

In [14]:
proj = nn.Linear(embed_dim_64, vocab_size_4)

In [ ]:
logits = proj(hidden)
logits.shape #(13, 4)

In [ ]:
predicted = F.softmax(logits, dim=-1)
predicted.shape

In [16]:
import wandb
from datetime import datetime



optim = torch.optim.Adam(list(embeddings_matrix.parameters()) + 
                         list(W_Q.parameters()) + 
                         list(W_K.parameters()) + 
                         list(W_V.parameters()) + 
                         list(ff.parameters()) + 
                         list(proj.parameters()), 
                         lr=0.001)

scheduler = torch.optim.lr_scheduler.StepLR(optim, step_size=30, gamma=0.1)

In [17]:

num_of_epochs = 500
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
wandb.init(project="mlx5.4-transformers", name=f"gpt_{timestamp}")

for epoch in range(num_of_epochs):
    # for input, actual in zip(inputs, actuals):
    print(f"Epoch {epoch}")
    seq_vectors = embeddings_matrix(input)
    q = W_Q(seq_vectors)
    k = W_K(seq_vectors)
    v = W_V(seq_vectors)
    atn = q @ k.T
    atn = atn / math.sqrt(embed_dim_64) # Not in Bes's code
    neginf = torch.full_like(atn, float('-inf'))
    mask = torch.triu(neginf, diagonal=1)
    atn_masked = atn + mask
    atn_probs = F.softmax(atn_masked, dim=-1) # softmax along rows
    out = atn_probs @ v
    hidden = ff(out)
    logits = proj(hidden)
    # predicted = F.softmax(logits, dim=-1)
    loss = F.cross_entropy(logits, actual)
    optim.zero_grad()
    loss.backward()
    optim.step()
    wandb.log({"loss": loss.item()})
    print(f"Loss: {loss.item()}")



wandb: Using wandb-core as the SDK backend. Please refer to https://wandb.me/wandb-core for more information.
wandb: Currently logged in as: learesong (learesong-machine-learning-institution). Use `wandb login --relogin` to force relogin


Epoch 0
Loss: 1.384610652923584
Epoch 1
Loss: 1.3216133117675781
Epoch 2
Loss: 1.2618681192398071
Epoch 3
Loss: 1.2021666765213013
Epoch 4
Loss: 1.1421043872833252
Epoch 5
Loss: 1.0824894905090332
Epoch 6
Loss: 1.024390697479248
Epoch 7
Loss: 0.9689090251922607
Epoch 8
Loss: 0.9153457880020142
Epoch 9
Loss: 0.8626354932785034
Epoch 10
Loss: 0.813799262046814
Epoch 11
Loss: 0.7742176651954651
Epoch 12
Loss: 0.7355626821517944
Epoch 13
Loss: 0.6970198750495911
Epoch 14
Loss: 0.669173002243042
Epoch 15
Loss: 0.6389105916023254
Epoch 16
Loss: 0.6117262244224548
Epoch 17
Loss: 0.5917384624481201
Epoch 18
Loss: 0.5641714930534363
Epoch 19
Loss: 0.54558265209198
Epoch 20
Loss: 0.5185415744781494
Epoch 21
Loss: 0.4961065649986267
Epoch 22
Loss: 0.46656784415245056
Epoch 23
Loss: 0.43823862075805664
Epoch 24
Loss: 0.40841323137283325
Epoch 25
Loss: 0.3814027011394501
Epoch 26
Loss: 0.3518151640892029
Epoch 27
Loss: 0.331540584564209
Epoch 28
Loss: 0.3133845031261444
Epoch 29
Loss: 0.27933639287

In [18]:
test_input = torch.tensor([0,1,1,2,2,3,3,1,1,2,2,3,3])

seq_vectors = embeddings_matrix(test_input)
q = W_Q(seq_vectors)
k = W_K(seq_vectors)
v = W_V(seq_vectors)
atn = q @ k.T
atn = atn / math.sqrt(embed_dim_64) # Not in Bes's code
neginf = torch.full_like(atn, float('-inf'))
mask = torch.triu(neginf, diagonal=1)
atn_masked = atn + mask
atn_probs = F.softmax(atn_masked, dim=-1) # softmax along rows
out = atn_probs @ v
hidden = ff(out)
logits = proj(hidden)
predicted = F.softmax(logits, dim=-1)

predicted_idx = torch.argmax(predicted, dim=-1).tolist()
print(predicted_idx)





[1, 1, 2, 2, 3, 3, 1, 1, 2, 2, 3, 3, 1]


In [67]:
loss = F.cross_entropy(predicted, actual)

In [ ]:
optim = torch.optim.Adam()